In [23]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import losses
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset

model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda:0")
loss = losses.MultipleNegativesRankingLoss(model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
import sqlite3

class SQL_Manager:
    def __init__(self):
        self.con = sqlite3.connect(
            "/kaggle/input/datasets/nhminh107/data-vietembed-rag/general_export.db"
        )
        self.con.row_factory = sqlite3.Row

    def get_general_data_batches(self, batch_size=10_000):
        cursor = self.con.cursor()

        cursor.execute("""
            SELECT *
            FROM general
            ORDER BY data_id ASC
        """)

        while True:
            rows = cursor.fetchmany(batch_size)

            if not rows:
                break

            yield [dict(row) for row in rows]

        cursor.close()


sql_mng = SQL_Manager()

In [25]:
import sqlite3
from datasets import IterableDataset, Features, Value

DB_PATH = "/kaggle/input/datasets/nhminh107/data-vietembed-rag/general_export.db"


def get_general_data(batch_size=10_000):
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row

    cursor = con.cursor()

    cursor.execute("""
        SELECT
            anchor,
            positive,
            hard_negative
        FROM general
        WHERE anchor IS NOT NULL
          AND positive IS NOT NULL
          AND hard_negative IS NOT NULL
        ORDER BY data_id ASC
    """)

    while True:
        rows = cursor.fetchmany(batch_size)

        if not rows:
            break

        for row in rows:
            yield {
                "anchor": row["anchor"],
                "positive": row["positive"]
            }

    cursor.close()
    con.close()


features = Features({
    "anchor": Value("string"),
    "positive": Value("string")
})


train_dataset = IterableDataset.from_generator(
    get_general_data,
    features=features,
    gen_kwargs={"batch_size": 10_000},
)

In [26]:
num_records = sql_mng.con.execute("""
    SELECT COUNT(*)
    FROM general
    WHERE anchor IS NOT NULL
      AND positive IS NOT NULL
""").fetchone()[0]

print("Records:", num_records)

Records: 3862671


In [30]:
import torch
import math
batch_size = 32
epochs = 2
gradient_accumulation_steps = 1

num_gpus = max(torch.cuda.device_count(), 1)

effective_batch_size = (
    batch_size
    * num_gpus
    * gradient_accumulation_steps
)

steps_per_epoch = math.ceil(
    num_records / effective_batch_size
)

max_steps = steps_per_epoch * epochs

print("GPUs:", num_gpus)
print("Effective batch size:", effective_batch_size)
print("Steps / epoch:", steps_per_epoch)
print("Max steps:", max_steps)

GPUs: 2
Effective batch size: 64
Steps / epoch: 60355
Max steps: 120710


In [31]:
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir="/kaggle/working/VietEmbed-RAG",

    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    max_steps=max_steps,

    learning_rate=2e-5,
    warmup_ratio=0.1,

    fp16=True,

    logging_steps=100,
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,
)

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


In [32]:
from sentence_transformers import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)
trainer.train()

ValueError: Batch does not contain any data (`None`). At the end of all iterable data available before expected stop iteration.